# A0 – Naive RAG Baseline

- **Adapted from:** `all_rag_techniques/simple_rag.ipynb`
- **Experiment ID:** `A0_NAIVE`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Establish baseline with fixed-size chunking, dense-only retrieval, fixed Top-K, single-pass generation, no query enhancement, no reranking, no compression, no citation verification.

## Hypothesis

Naive RAG provides a functional but limited pipeline. It should score lower than Advanced RAG on Recall@K, Faithfulness, and Citation Correctness, while having lower latency.

## 1. Setup & Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, build_result_record, Timer

In [ ]:
# Load and display config
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A0_NAIVE"
NOTEBOOK = "01_naive_rag.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]

In [ ]:
# Build models
llm = build_llm(config)
embeddings = build_embeddings(config)

# Quick connection test
test_response = llm.invoke("Say hello in one word.")
print(f"LLM OK: {test_response.content[:50]}")

test_vec = embeddings.embed_query("test")
print(f"Embedding OK: dim={len(test_vec)}")

## 2. Load Corpus

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Load documents from report_data/raw
raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
print(f"Loading documents from: {raw_data_path}")

# Support PDF and text files
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from {len(list(raw_data_path.glob('*.pdf')))} PDF files.")

## 3. Fixed-Size Chunking (Baseline)

In [ ]:
chunk_size = config["baseline"]["chunk_size"]
chunk_overlap = config["baseline"]["chunk_overlap"]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

# Clean tabs
for chunk in chunks:
    chunk.page_content = chunk.page_content.replace('\t', ' ')

print(f"Created {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap})")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

## 4. Create Vector Store & Retriever

In [ ]:
# Create FAISS index
vectorstore = FAISS.from_documents(chunks, embeddings)

# Fixed Top-K retriever (no reranking, no hybrid)
top_k = config["baseline"]["top_k"]
retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

print(f"Vector store created with {vectorstore.index.ntotal} vectors.")
print(f"Retriever: dense-only, Top-K={top_k}")

## 5. Single Query Demo

In [ ]:
from langchain_core.prompts import PromptTemplate

# Simple baseline prompt - no grounding, no citation requirement
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)

chain = NAIVE_PROMPT | llm

# Test query
test_query = "What is the main cause of climate change?"
docs = retriever.invoke(test_query)
context = "\n\n".join([d.page_content for d in docs])
response = chain.invoke({"context": context, "question": test_query})

print(f"Query: {test_query}")
print(f"Answer: {response.content}")
print(f"\nSources: {[d.metadata.get('source', 'unknown') for d in docs]}")

## 6. Run Full Evaluation Set

In [ ]:
import json

# Load evaluation questions
questions_path = PROJECT_ROOT / config["paths"]["questions"]
with open(questions_path, "r", encoding="utf-8") as f:
    eval_questions = json.load(f)

print(f"Loaded {len(eval_questions)} evaluation questions.")

In [ ]:
results = []

for q in eval_questions:
    qid = q["question_id"]
    question = q["question"]
    relevant_docs = q.get("relevant_documents", [])

    # Retrieval
    with Timer() as t_retrieval:
        docs = retriever.invoke(question)

    retrieved_ids = [d.metadata.get("source", f"chunk_{i}") for i, d in enumerate(docs)]
    context = "\n\n".join([d.page_content for d in docs])

    # Generation
    with Timer() as t_generation:
        response = chain.invoke({"context": context, "question": question})

    answer = response.content

    # Compute retrieval metrics
    metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=top_k)

    # Build result record
    record = build_result_record(
        experiment_id=EXPERIMENT_ID,
        notebook=NOTEBOOK,
        config_hash=CONFIG_HASH,
        seed=SEED,
        question_id=qid,
        question=question,
        retrieved_documents=[
            {"content": d.page_content[:200], "metadata": d.metadata}
            for d in docs
        ],
        answer=answer,
        predicted_abstain=False,
        latency={
            "retrieval_seconds": t_retrieval.elapsed,
            "generation_seconds": t_generation.elapsed,
            "total_seconds": t_retrieval.elapsed + t_generation.elapsed,
        },
        metrics=metrics,
    )
    results.append(record)
    print(f"  [{qid}] {t_retrieval.elapsed + t_generation.elapsed:.2f}s")

print(f"\nCompleted {len(results)} questions.")

## 7. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]

# Save JSONL
save_jsonl(results, output_dir / "A0_naive.jsonl")

# Save config snapshot
save_config_snapshot(config, output_dir)

## 8. Summary Metrics

In [ ]:
import numpy as np

# Aggregate metrics
metric_keys = [k for k in results[0]["metrics"].keys() if results[0]["metrics"][k] is not None]

print("=" * 60)
print(f"A0 NAIVE RAG BASELINE RESULTS ({len(results)} questions)")
print("=" * 60)

for key in metric_keys:
    values = [r["metrics"][key] for r in results if r["metrics"].get(key) is not None]
    if values:
        print(f"  {key:20s}: mean={np.mean(values):.3f}  std={np.std(values):.3f}")

# Latency
latencies = [r["latency"]["total_seconds"] for r in results]
print(f"\n  {'avg_latency':20s}: {np.mean(latencies):.3f}s")
print(f"  {'p95_latency':20s}: {np.percentile(latencies, 95):.3f}s")
print("=" * 60)

## 9. Failure Cases Analysis

In [ ]:
# Show questions with hit_rate = 0 (retrieval missed relevant docs)
failures = [r for r in results if r["metrics"].get(f"hit_rate_at_{top_k}", 1) == 0]

print(f"\nRetrieval failures (hit_rate=0): {len(failures)}/{len(results)}")
for f in failures[:5]:
    print(f"  [{f['question_id']}] {f['question'][:80]}")